In [1]:
# Client setup
# Anthropic doesn't offer an embeddings endpoint — its docs recommend Voyage AI
# as the embeddings provider. The `voyageai` package is pinned in
# requirements.txt, and the client reads VOYAGE_API_KEY from the .env file.
from dotenv import load_dotenv
import voyageai

load_dotenv()

client = voyageai.Client()

C:\Users\Juan Camilo\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Chunk by section — the strategy chosen in notebook 013
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [3]:
# Embedding generation
# `voyage-4` is the current general-purpose model (32K-token context window,
# 1024 dimensions by default); `voyage-4-lite` optimizes cost/latency and
# `voyage-4-large` retrieval quality.
#
# `input_type` prepends a retrieval-specific prompt before embedding:
# "document" for the texts being indexed, "query" for the question used to
# search. Voyage recommends setting it for retrieval — embeddings generated
# with and without it remain compatible with each other.
def generate_embedding(text, model="voyage-4", input_type=None):
    result = client.embed([text], model=model, input_type=input_type)

    return result.embeddings[0]

In [4]:
# An embedding places a text in a semantic space where similar meanings end up
# close together. Inspect its shape instead of dumping all 1024 numbers.
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

embedding = generate_embedding(chunks[0], input_type="document")
print(f"Dimensions: {len(embedding)}")
print(f"First values: {embedding[:5]}")

Dimensions: 1024
First values: [-0.03152728080749512, -0.032203830778598785, -0.009268750436604023, 0.009877646341919899, -0.03382755443453789]


In [5]:
# Semantic search in a nutshell: embed all sections in one API call (the
# endpoint accepts up to 1,000 texts per request) and compare them against a
# question. Voyage embeddings are normalized to length 1, so the dot product
# is the cosine similarity — higher means semantically closer. The top matches
# are the sections that discuss security incidents, even though the question
# shares almost no keywords with them.
result = client.embed(chunks, model="voyage-4", input_type="document")
chunk_embeddings = result.embeddings

query_embedding = generate_embedding(
    "Were there any security breaches this year?", input_type="query"
)

similarities = [
    sum(q * c for q, c in zip(query_embedding, chunk_embedding))
    for chunk_embedding in chunk_embeddings
]

ranked = sorted(zip(similarities, chunks), reverse=True)
for similarity, chunk in ranked[:3]:
    print(f"{similarity:.4f}  {chunk.splitlines()[0]}")

0.3031  Executive Summary
0.2907  Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011
0.2799  Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts
